# reverse-l1b — synthetic RAW L0 vs original ESA RAW L0

Visual comparison of the `reverse-l1b` phase output (**synthetic** L0 generated from the real ESA
**L1B** digital counts) against the **original ESA L0** for the 2024-04-08 S2B PPB datatake.

Shows, per band: side-by-side raw images (synthetic | real | difference), a per-band statistics
table, and a PRNU/texture zoom. Attach a kernel with `s2_msi_raw_generator`, `zarr` and
`matplotlib` installed, with the run env set (`S2_E2ES_L1B`, `S2_E2ES_GIPP_DIR`,
`S2_E2ES_EQOG_ADF`, and optionally `S2_E2ES_L0_DIR`).


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import zarr
from s2_msi_raw_generator import l0product, gipp
from s2_msi_raw_generator import forward_radiometric_atbd as fwd

DET = 5
LINE_OFFSET = 3500   # real L0 leads L1B by ~3500 lines for this datatake (probe_align)

# synthetic L0 = newest reverse-l1b output under S2_E2ES_L0_DIR (or outputs/L0)
_syn_dir = os.environ.get("S2_E2ES_L0_DIR") or os.path.expanduser(
    "~/data-store/synthetic-raw-generated/outputs/L0")
SYN = sorted(glob.glob(os.path.join(_syn_dir, "S02MSIL0__*_reverse.zarr")))[-1]
# original ESA L0 for the same datatake (next to the L1B)
REAL = os.path.dirname(os.environ["S2_E2ES_L1B"]) + \
    "/S02MSIL0__20240408T053621_0566_B105_TC7D.zarr"

gs = gipp.load_gipp_set(os.environ["S2_E2ES_GIPP_DIR"], eqog_adf=os.environ["S2_E2ES_EQOG_ADF"])
g_real = zarr.open_group(REAL, mode="r")
print("synthetic:", SYN)
print("real     :", REAL)
print("bands    :", list(zarr.open_group(SYN, mode="r")[f"measurements/d{DET:02d}"].group_keys()))


In [ ]:
def load_pair(band, n_lines=2048):
    """Line-aligned, blind-stripped active DN for one band/detector: (synthetic, real)."""
    syn = l0product.read_l0_isp_dn(SYN, DET, band).astype(np.float64)
    h, w = syn.shape
    real = np.asarray(
        g_real[f"measurements/d{DET:02d}/{band.lower()}/img"][LINE_OFFSET:LINE_OFFSET + h, :w],
        dtype=np.float64,
    )
    blind = gs.blind.get(band, {}).get(DET)          # BLINDP → strip blind columns
    if blind is not None and len(blind) and max(blind) < w and (w - len(blind)) > 0:
        keep = np.setdiff1d(np.arange(w), np.asarray(blind, int))
        syn, real = syn[:, keep], real[:, keep]
    return syn[:n_lines], real[:n_lines]


def stretch(a, ref=None, lo=2, hi=98):
    p1, p2 = np.percentile(ref if ref is not None else a, [lo, hi])
    return np.clip((a - p1) / (p2 - p1 + 1e-9), 0, 1)


## Side-by-side raw images (synthetic | real ESA | difference)

In [ ]:
BANDS = ["B03", "B04", "B08", "B11"]
N = 2048
fig, ax = plt.subplots(len(BANDS), 3, figsize=(13, 3.6 * len(BANDS)))
for i, bn in enumerate(BANDS):
    syn, real = load_pair(bn, N)
    ax[i, 0].imshow(stretch(syn, ref=real), cmap="gray", aspect="auto")
    ax[i, 0].set_title(f"{bn}  synthetic RAW (reverse-l1b)")
    ax[i, 1].imshow(stretch(real, ref=real), cmap="gray", aspect="auto")
    ax[i, 1].set_title(f"{bn}  original ESA RAW")
    d = syn - real
    lim = np.percentile(np.abs(d), 98) + 1e-6
    ax[i, 2].imshow(d, cmap="RdBu_r", aspect="auto", vmin=-lim, vmax=lim)
    ax[i, 2].set_title(f"{bn}  diff (syn-real)  med={np.median(d):+.1f} DN")
    for a in ax[i]:
        a.axis("off")
plt.tight_layout()
plt.savefig("reverse_l1b_compare.png", dpi=110, bbox_inches="tight")
plt.show()


## Per-band statistics (median, column FPN, RMSE)

In [ ]:
print(f"{'band':4} {'syn_med':>8} {'real_med':>8} {'d%':>6} {'syn_fpn':>8} {'real_fpn':>8} {'rmse':>7}")
for bn in ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]:
    try:
        syn, real = load_pair(bn, 4096)
    except Exception as e:
        print(f"{bn:4} skip ({type(e).__name__})"); continue
    d = 100 * (np.median(syn) - np.median(real)) / max(np.median(real), 1e-6)
    rmse = float(np.sqrt(np.mean((syn - real) ** 2)))
    print(f"{bn:4} {np.median(syn):8.1f} {np.median(real):8.1f} {d:6.1f} "
          f"{fwd.column_fpn(syn):8.3f} {fwd.column_fpn(real):8.3f} {rmse:7.1f}")


## PRNU / texture zoom (do the fixed-pattern stripes match?)

In [ ]:
bn = "B03"
syn, real = load_pair(bn, 20000)
cy, cx, cs = 8000, 1000, 220
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
for a, img, t in zip(ax, [syn, real], ["synthetic (reverse-l1b)", "original ESA"]):
    crop = img[cy:cy + cs, cx:cx + cs]
    a.imshow(stretch(crop, ref=real[cy:cy + cs, cx:cx + cs]), cmap="gray")
    a.set_title(f"{bn}  {t}  (zoom {cs}x{cs})")
    a.axis("off")
plt.tight_layout()
plt.show()
